# TreeSimplify End-to-End Demo

Full pipeline: beam search + BigInt rational post-processing + multi-pass convergence.

Key features:
- **BigInt rationals**: all expressions parsed with `BigInt(N)//BigInt(D)` to avoid Int64 overflow in polynomial arithmetic
- **Post-processing**: `SymbolicUtils.simplify_fractions` applied to bounded subtrees after beam search
- **Multi-pass convergence**: recursive passes with growing subtree caps (2.5× per pass, up to 3 passes)
- Compares TreeSimplify output vs expected corpus scores

In [1]:
import Pkg
Pkg.activate(joinpath(pwd(), ".."))
Pkg.instantiate()

  Activating project at `~/src/TreeSimplify.jl`
Precompiling packages...
   4470.8 ms  ✓ TreeSimplify
  1 dependency successfully precompiled in 5 seconds. 162 already precompiled.


In [2]:
using Symbolics, Latexify, Printf, Random
import TreeSimplify
import TreeSimplify.SymbolicUtils

println("Loaded.")

Loaded.


## Configuration

Default multi-pass pipeline with BigInt rationals.

In [3]:
config = TreeSimplify.RunConfig()  # uses defaults with BigInt + multi-pass
println("Budget:   max_depth=", config.budget.max_depth,
        " beam_width=", config.budget.beam_width,
        " max_time=", config.budget.max_time_seconds, "s")
println("Post:     max_nodes=", config.post_simplify_max_nodes,
        " timeout=", config.post_simplify_timeout_secs, "s")
println("Passes:   max=", config.simplify_max_passes,
        " growth=", config.simplify_pass_nodes_growth, "×")

Budget:   max_depth=16 beam_width=64 max_time=5.0s
Post:     max_nodes=800 timeout=60.0s
Passes:   max=3 growth=2.5×


## Full Corpus Results

Benchmark corpus of large rational expressions with expected compact forms.
Comparing TreeSimplify output score vs expected compact-form score.

In [4]:
function corpus_benchmark()
    sw_path = normpath(joinpath(@__DIR__, "..", "expressions",
        "sw_nonrwa_order4_coeffs.txt"))
    exp_path = normpath(joinpath(@__DIR__, "..", "expressions",
        "extracted_sw_nonrwa_coefficients_output.txt"))
    sw    = TreeSimplify._load_sw_sections(sw_path)
    expec = TreeSimplify._load_expected_sections(exp_path)

    # Run end-to-end validation (input ≡ expected, output ≡ expected)
    e2e = TreeSimplify.run_end_to_end_validation(;
        sw_path = sw_path, expected_path = exp_path, config = config)

    # Collect output scores
    println(rpad("Section", 10), lpad("In", 6), lpad("Out", 6),
            lpad("Exp", 6), lpad("Ratio", 7), rpad("Time", 8),
            "In≡Exp  Out≡Exp")
    println(repeat("-", 70))

    total_ratio = 0.0
    for label in sort!(collect(keys(sw)))
        t = @elapsed result = TreeSimplify.simplify(sw[label]; config = config)
        exp_score = TreeSimplify.expression_score(
            TreeSimplify.expression_term(expec[label]), config.scoring)
        ratio = result.score_after / exp_score
        total_ratio += ratio

        rec = [r for r in e2e.records if r.label == label][1]
        in_eq  = rec.input_equivalent_to_expected  ? "✓" : "✗"
        out_eq = rec.output_equivalent_to_expected ? "✓" : "✗"

        @printf "%-10s %6d %6d %6d %6.2f× %6.1fs  %s     %s\n" (
            label,
            round(Int, result.score_before),
            round(Int, result.score_after),
            round(Int, exp_score),
            ratio, t, in_eq, out_eq)
    end

    avg = total_ratio / 9
    @printf "\nAverage: %.2f× (%.0f%% from target)\n" avg (avg*100-100)
end
corpus_benchmark()

LoadError: MethodError: no method matching run_end_to_end_validation(::String, ::String; sw_path::String, expected_path::String, config::TreeSimplify.RunConfig)
This method does not support all of the given keyword arguments (and may not support any).

[0mClosest candidates are:
[0m  run_end_to_end_validation(::String, ::String; config)[91m got unsupported keyword arguments "sw_path", "expected_path"[39m
[0m[90m   @[39m [36mTreeSimplify[39m [90m~/src/TreeSimplify.jl/src/[39m[90m[4mbenchmarks.jl:257[24m[39m
[0m  run_end_to_end_validation(::String; ...)
[0m[90m   @[39m [36mTreeSimplify[39m [90m~/src/TreeSimplify.jl/src/[39m[90m[4mbenchmarks.jl:257[24m[39m
[0m  run_end_to_end_validation(; ...)
[0m[90m   @[39m [36mTreeSimplify[39m [90m~/src/TreeSimplify.jl/src/[39m[90m[4mbenchmarks.jl:257[24m[39m


  Julia 1.12 has introduced more strict world age semantics for global bindings.
  !!! This code may malfunction under Revise.
  !!! This code will error in future versions of Julia.
Hint: Add an appropriate `invokelatest` around the access to this binding.
To make this warning an error, and hence obtain a stack trace, use `julia --depwarn=error`.


## Hard Test Cases (Regression)

Domain-agnostic algebraic stress tests.

In [5]:
@variables x y z a b c u v w
cases = [
    ("Cyclic diff-quotient sum",
        ((x^2-y^2)/(x-y)) + ((y^2-z^2)/(y-z)) + ((z^2-x^2)/(z-x))),
    ("Nested rational 3-cycle",
        (1+1/(1+1/(1+x+y))) + (1+1/(1+1/(1+y+z))) + (1+1/(1+1/(1+z+x)))),
    ("High-degree parity cancel",
        (x+y+z)^7 - (x-y-z)^7 + (x+y-z)^7 - (x-y+z)^7),
    ("Squared cyclic diff-quotients",
        ((x^2-y^2)/(x-y))^2 + ((y^2-z^2)/(y-z))^2 + ((z^2-x^2)/(z-x))^2),
    ("CSE-heavy repeated block",
        ((a+b+c)^2 - (a-b-c)^2)*3),
    ("Rewrite-maze neutral elts",
        (((u+0)*1)/1 + ((v+0)*1)/1 + ((w+0)*1)/1) + (u+u) + (v*1) + (w*1)),
    ("Nested cancellation islands",
        (((a+b)/(a+b)) + ((b+c)/(b+c)) + ((c+a)/(c+a)))*1 + 0),
]
println("Loaded ", length(cases), " test cases.")

Loaded 7 test cases.


In [6]:
for (lbl, expr) in cases
    t = @elapsed r = TreeSimplify.simplify(expr; config = config)
    eq = TreeSimplify.validate_equivalence(expr, r.best_expr, config)
    len_in  = length(TreeSimplify.stable_serialize(expr))
    len_out = length(TreeSimplify.stable_serialize(r.best_expr))

    println(rpad(lbl, 35),
        "  score ", round(Int, r.score_before), " → ", rpad(round(Int, r.score_after), 5),
        "  len ", len_in, " → ", len_out,
        "  ", round(t, digits=2), "s",
        "  eq=", eq.passed,
        "  accepted=", r.accepted)
end

Cyclic diff-quotient sum             score 80 → 12     len 77 → 12  28.31s  eq=true  accepted=true
Nested rational 3-cycle              score 91 → 91     len 85 → 85  1.5s  eq=true  accepted=false
High-degree parity cancel            score 50 → 50     len 66 → 66  0.16s  eq=true  accepted=false
Squared cyclic diff-quotients        score 89 → 20     len 89 → 33  0.39s  eq=true  accepted=true
CSE-heavy repeated block             score 26 → 26     len 35 → 35  0.0s  eq=true  accepted=false
Rewrite-maze neutral elts            score 12 → 12     len 12 → 12  0.0s  eq=true  accepted=false
Nested cancellation islands          score 1 → 1      len 1 → 1  0.18s  eq=true  accepted=false


## Determinism Check

Two runs with same config → identical structural hash.

In [7]:
det_ok = true
for (lbl, expr) in cases
    rA = TreeSimplify.simplify(expr; config = config)
    rB = TreeSimplify.simplify(expr; config = config)
    hA = TreeSimplify.structural_hash(rA.best_expr)
    hB = TreeSimplify.structural_hash(rB.best_expr)
    same = (hA == hB) && (rA.score_after == rB.score_after)
    println(rpad(lbl, 35), " det=", same)
    det_ok &= same
end
println("\nDeterminism passed: ", det_ok)

Cyclic diff-quotient sum            det=true
Nested rational 3-cycle             det=true
High-degree parity cancel           det=true
Squared cyclic diff-quotients       det=true
CSE-heavy repeated block            det=true
Rewrite-maze neutral elts           det=true
Nested cancellation islands         det=true

Determinism passed: true


## Fuzz Sweep

Random algebraic expressions with validation.

In [8]:
rng = MersenneTwister(0x1234)
vars = [x, y, z]
rand_leaf(rng, vars) = rand(rng) < 0.6 ? vars[rand(rng, 1:length(vars))] : rand(rng, -3:3)

function rand_expr(rng, vars, depth)
    depth <= 0 && return rand_leaf(rng, vars)
    op = rand(rng, [:+, :-, :*, :/])
    a = rand_expr(rng, vars, depth - 1)
    b = rand_expr(rng, vars, depth - 1)
    local b_val = Symbolics.value(b)
    if b_val isa Number && iszero(b_val)
        b = Num(1)
    end
    expr = op === :+ ? (a + b) : op === :- ? (a - b) : op === :* ? (a * b) : (a / b)
    rand(rng) < 0.4 && (expr = (expr + 0) * 1)
    rand(rng) < 0.25 && (expr = expr / 1)
    return expr
end

N = 40
valid = accepted = improved = 0
for i in 1:N
    expr = rand_expr(rng, vars, 3)
    r = TreeSimplify.simplify(expr; config = config)
    rep = TreeSimplify.validate_equivalence(expr, r.best_expr, config)
    valid    += rep.passed ? 1 : 0
    accepted += r.accepted  ? 1 : 0
    improved += (r.score_after < r.score_before) ? 1 : 0
end
println("Runs:       ", N)
println("Validated:  ", valid, "/", N)
println("Accepted:   ", accepted, "/", N)
println("Improved:   ", improved, "/", N)

Runs:       40
Validated:  40/40
Accepted:   1/40
Improved:   1/40
